# Chapter 03: Missing Data

Real-world datasets almost always contain missing values. Pandas uses several sentinel values to represent missing data and provides a comprehensive toolkit for detecting, removing, and filling gaps.

In this notebook we will cover:

- `NaN`, `None`, and `pd.NA` — what they are and how they differ
- `isnull()`, `notnull()`, `isna()`, `notna()`
- `dropna()` with `axis`, `how`, `thresh`, and `subset`
- `fillna()` with scalar, method (`ffill`/`bfill`), and per-column dict
- `interpolate()`
- Impact of missing values on aggregations

In [ ]:
import pandas as pd
import numpy as np

## What Are Missing Values?

| Sentinel | Type | Notes |
|----------|------|-------|
| `np.nan` | `float` | The traditional missing marker. Forces integer columns to float. |
| `None` | `NoneType` | Python's built-in null. Pandas converts it to `NaN` in numeric contexts. |
| `pd.NA` | `pandas.NA` | Introduced in pandas 1.0 for nullable integer/boolean/string dtypes. |

In [ ]:
# np.nan is a float
print(type(np.nan))

# Integers become float when NaN is present
s = pd.Series([1, 2, np.nan, 4])
print(s)
print(f"dtype: {s.dtype}")

In [ ]:
# pd.NA preserves integer dtype with nullable Int64
s_nullable = pd.Series([1, 2, pd.NA, 4], dtype='Int64')
print(s_nullable)
print(f"dtype: {s_nullable.dtype}")

## Working with a Real Dataset

The `movie_scores.csv` file contains intentional missing values.

In [ ]:
movies = pd.read_csv('data/movie_scores.csv')
movies

## Detecting Missing Values

- `isnull()` / `isna()` — returns `True` where values are missing (these are aliases)
- `notnull()` / `notna()` — returns `True` where values are present

In [ ]:
# Boolean mask of missing values
movies.isnull()

In [ ]:
# Count missing values per column
movies.isnull().sum()

In [ ]:
# Percentage of missing values per column
(movies.isnull().sum() / len(movies) * 100).round(1)

In [ ]:
# Find rows where a specific column is missing
movies[movies['pre_movie_score'].isna()]

In [ ]:
# Find rows where ALL values are present
movies[movies.notna().all(axis=1)]

## `dropna()` — Removing Missing Values

Key parameters:
- `axis=0` (default): drop **rows** with missing values
- `axis=1`: drop **columns** with missing values
- `how='any'` (default): drop if **any** value is missing
- `how='all'`: drop only if **all** values are missing
- `thresh=N`: keep rows with at least N non-null values
- `subset`: only consider specific columns

In [ ]:
# Drop rows where ANY value is missing
movies.dropna()

In [ ]:
# Drop rows where ALL values are missing (keeps partial rows)
movies.dropna(how='all')

In [ ]:
# Keep rows with at least 4 non-null values
movies.dropna(thresh=4)

In [ ]:
# Only consider certain columns for the null check
movies.dropna(subset=['first_name', 'last_name'])

In [ ]:
# Drop columns (axis=1) that have any missing values
movies.dropna(axis=1)

## `fillna()` — Filling Missing Values

Instead of dropping data, you can substitute missing values.

In [ ]:
# Fill all NaN with a scalar
movies.fillna(0)

In [ ]:
# Fill with a string for text columns
movies['first_name'].fillna('Unknown')

In [ ]:
# Fill with different values per column using a dictionary
fill_values = {
    'first_name': 'Unknown',
    'last_name': 'Unknown',
    'age': movies['age'].mean(),
    'pre_movie_score': movies['pre_movie_score'].median(),
    'post_movie_score': movies['post_movie_score'].median()
}
movies.fillna(fill_values)

In [ ]:
# Forward fill (ffill): propagate the last valid value forward
s = pd.Series([1.0, np.nan, np.nan, 4.0, np.nan, 6.0])
print("Original:      ", s.values)
print("Forward filled:", s.ffill().values)

In [ ]:
# Backward fill (bfill): propagate the next valid value backward
print("Backward filled:", s.bfill().values)

## `interpolate()` — Estimating Missing Values

Interpolation estimates missing values based on surrounding data points. The default is linear interpolation.

In [ ]:
s = pd.Series([0.0, np.nan, np.nan, 6.0, np.nan, 10.0])
print("Original:     ", s.values)
print("Interpolated: ", s.interpolate().values)

In [ ]:
# Interpolation on a time-like index
ts = pd.Series(
    [100, np.nan, np.nan, np.nan, 200, np.nan, 150],
    index=pd.date_range('2024-01-01', periods=7, freq='D')
)
print("Original:")
print(ts)
print("\nInterpolated:")
print(ts.interpolate())

## Impact on Aggregations

Pandas aggregation functions (`mean`, `sum`, `std`, etc.) skip `NaN` by default. This is usually what you want, but you should be aware of it.

In [ ]:
s = pd.Series([10, 20, np.nan, 40, 50])

print(f"mean (NaN skipped):   {s.mean()}")      # (10+20+40+50)/4 = 30.0
print(f"sum  (NaN skipped):   {s.sum()}")        # 120.0
print(f"count (non-null):     {s.count()}")      # 4

In [ ]:
# Be aware: len() counts ALL elements including NaN, count() does not
print(f"len(s):    {len(s)}")
print(f"s.count(): {s.count()}")

In [ ]:
# Operations with NaN propagate NaN
print(f"np.nan + 5 = {np.nan + 5}")
print(f"np.nan > 3 = {np.nan > 3}")
print(f"np.nan == np.nan = {np.nan == np.nan}")  # NaN is not equal to itself!

> **Key insight**: To check for NaN, always use `pd.isna()` or `.isna()` — never use `== np.nan` because NaN is not equal to itself.

## Summary

In this notebook we covered missing data handling in pandas:

- Missing values are represented by `np.nan` (float), `None`, or `pd.NA` (nullable types).
- Detect with `isna()` / `isnull()` and `notna()` / `notnull()`.
- `dropna()` removes rows or columns; control behaviour with `how`, `thresh`, and `subset`.
- `fillna()` substitutes missing values with scalars, forward/backward fill, or per-column dicts.
- `interpolate()` estimates missing values based on surrounding data.
- Aggregations skip NaN by default; use `count()` (not `len()`) to get non-null counts.
- Never compare with `== np.nan` — use `pd.isna()` instead.

Next up: **GroupBy and Aggregation** — split-apply-combine operations.